# 下游诊断：强效基因恢复与扰动身份检索
两项任务只用冻结验证数据做诊断，不用于选择 checkpoint 或调参。所有数值来自服务端的条件级标量汇总。

## 任务 A：非靶基因的强响应恢复
固定 K=20/50，按真值和预测的绝对表达变化分别排序，排除直接靶基因。查看 overlap precision / recall，以及在真值 top-K 上的符号准确率。复制对照预测无非零效应时 precision 无定义，recall 为零（真值有强响应时），未响应在符号评分里计错。这里的 top-K 只是效应大小排序，无显著性检验。方向诊断参考 [GEARS](https://cs.stanford.edu/people/jure/pubs/gears-natbio23.pdf)。

In [ ]:
from analysis_charts import downstream_retrieval, downstream_top, load, missing_diagnostics

data = load("condition-aggregates.json")
rows = data["rows"]
if data["response_diagnostics"] is not None:
    downstream_top(rows)
else:
    missing_diagnostics()

## 任务 B：从预测响应找回扰动身份
每个候选用同一冻结表达轴上相对其对照池的真值响应。查询用同一条件的对照池作中心，以余弦排序；报告 Top-1、Top-5、MRR 和秩分布，并比较复制对照、训练均值与随机水平。并列按随机排列的期望打分，零真值向量不进入检索库。训练均值可能借由条件特异的对照池携带信息，因此另画随机参考。模型分数若接近这些基线，优先检查预测是否对扰动身份不敏感。排序评估参考 [PerturBench](https://proceedings.neurips.cc/paper_files/paper/2025/hash/8aee537279a66ced96319dfca3c00002-Abstract-Datasets_and_Benchmarks_Track.html)。

In [ ]:
if data["response_diagnostics"] is not None:
    downstream_retrieval(rows, data["response_diagnostics"])